In [1]:
!pip -q install tokenizers transformers datasets scikit-learn matplotlib pandas tensorflow tensorflow_hub umap-learn


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Sample corpus we'll train our small tokenizers on.
# (In practice you'd use a much larger corpus - this is just for a quick, visualizable demo.)
corpus = [
    "The quick brown fox jumps over the lazy dog.",
    "Natural language processing is a fascinating field of artificial intelligence.",
    "Tokenization breaks text into smaller pieces called tokens.",
    "Subword tokenization handles rare and out-of-vocabulary words gracefully.",
    "The bank raised interest rates to control inflation.",
    "We sat by the river bank and watched the sunset.",
    "I need to bank this check before the branch closes.",
    "The bat flew out of the cave at dusk.",
    "He swung the bat and hit a home run.",
    "Deep learning models like BERT, GPT, and ELMo revolutionized NLP.",
    "Unhappiness and disagreement are common in unpredictable situations.",
    "Emoji like 😀 and 🌍 show up in real-world text all the time.",
]

with open("corpus.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(corpus))

corpus[:3]


## 1. BPE (Byte-Pair Encoding)

BPE starts from individual characters and **greedily merges the most frequent adjacent pair**
of symbols, over and over, until it hits the target vocab size. The merge rule is pure frequency.

In [ ]:
## 1. BPE (Byte-Pair Encoding)

from tokenizers import Tokenizer, models, trainers, pre_tokenizers

bpe_tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
bpe_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

bpe_trainer = trainers.BpeTrainer(
    vocab_size=300,
    min_frequency=1,
    special_tokens=["[UNK]", "[PAD]"]
)
bpe_tokenizer.train(["corpus.txt"], bpe_trainer)

print(f"BPE vocab size: {bpe_tokenizer.get_vocab_size()}")


In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

bpe_tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
bpe_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

bpe_trainer = trainers.BpeTrainer(
    vocab_size=300,
    min_frequency=1,
    special_tokens=["[UNK]", "[PAD]"]
)
bpe_tokenizer.train(["corpus.txt"], bpe_trainer)

print(f"BPE vocab size: {bpe_tokenizer.get_vocab_size()}")


In [ ]:
def show_tokens(tokenizer, sentence, name=""):
    enc = tokenizer.encode(sentence)
    print(f"[{name}] {sentence}")
    print("  tokens:", enc.tokens)
    print("  count :", len(enc.tokens))
    print()
    return enc.tokens

test_sentences = [
    "Unhappiness and disagreement are common.",
    "The bank raised interest rates.",
    "Tokenization is fascinating!",
]

bpe_outputs = [show_tokens(bpe_tokenizer, s, "BPE") for s in test_sentences]


## 2. WordPiece

WordPiece also merges pairs bottom-up, but instead of raw frequency it picks the pair that
**maximizes the likelihood of the training corpus**:

```
score(pair) = freq(pair) / (freq(first) * freq(second))
```

This favors pairs that co-occur *more than you'd expect by chance*, not just pairs that are common.
Non-initial subword pieces get a `##` prefix (BERT-style) so you can tell where a word was split.


In [ ]:
wp_tokenizer = Tokenizer(models.WordPiece(unk_token="[UNK]"))
wp_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

wp_trainer = trainers.WordPieceTrainer(
    vocab_size=300,
    min_frequency=1,
    special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]"]
)
wp_tokenizer.train(["corpus.txt"], wp_trainer)

print(f"WordPiece vocab size: {wp_tokenizer.get_vocab_size()}")
wp_outputs = [show_tokens(wp_tokenizer, s, "WordPiece") for s in test_sentences]


## 3. Byte-Level BPE (GPT-2 style)

Same merge *algorithm* as BPE, but the starting alphabet is the **256 raw bytes** instead of
Unicode characters. This means it can represent *any* text — emoji, any language, weird symbols —
with zero `[UNK]` tokens, because everything is bytes underneath.


In [ ]:
from tokenizers import ByteLevelBPETokenizer

blbpe_tokenizer = ByteLevelBPETokenizer()
blbpe_tokenizer.train(files=["corpus.txt"], vocab_size=300, min_frequency=1)

emoji_sentences = test_sentences + ["Emoji like 😀 and 🌍 show up in real text."]

for s in emoji_sentences:
    enc = blbpe_tokenizer.encode(s)
    print(f"[Byte-BPE] {s}")
    print("  tokens:", enc.tokens)
    print("  count :", len(enc.tokens))
    print()


## 4. Side-by-Side Comparison

In [ ]:
rows = []
for s in test_sentences:
    rows.append({
        "sentence": s,
        "BPE": " | ".join(show_tokens.__wrapped__(s) if False else bpe_tokenizer.encode(s).tokens),
        "WordPiece": " | ".join(wp_tokenizer.encode(s).tokens),
        "Byte-Level BPE": " | ".join(blbpe_tokenizer.encode(s).tokens),
    })

df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", None)
df


In [ ]:
# Token-count comparison chart
counts = {
    "BPE": [len(bpe_tokenizer.encode(s).tokens) for s in test_sentences],
    "WordPiece": [len(wp_tokenizer.encode(s).tokens) for s in test_sentences],
    "Byte-Level BPE": [len(blbpe_tokenizer.encode(s).tokens) for s in test_sentences],
}
count_df = pd.DataFrame(counts, index=[f"sent {i+1}" for i in range(len(test_sentences))])
count_df.plot(kind="bar", figsize=(8, 4))
plt.title("Token count per sentence, by tokenization method")
plt.ylabel("number of tokens")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
